<a href="https://colab.research.google.com/github/Pauloade123/ML-intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pauloade123/ML-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis (Grain):** Daily performance at the client and web-page level (`report_date` + `client_hash_id` + `content_hash_id`).
* **Time Window:** The historical date range present in the `fact_content_daily_performance` dataset (verified via min/max date queries).

In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
from google.colab import userdata
from datasets import load_dataset

hf_token = userdata.get('Flyrank_intern')
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=hf_token
    )
print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**1. Context / Primary Keys (Identifiers):**
* `report_date`, `client_hash_id`, `content_hash_id`
* *Role:* Uniquely defines the daily page-level grain across clients.

**2. Target / Label (Opportunity Metric):**
* `ga4_engaged_sessions`
* *Role:* Evaluates whether a page captures and retains active user engagement relative to incoming traffic volume.

**3. Features (Model & Scoring Inputs):**
* **Position & Visibility:** `gsc_impressions`, `gsc_avg_position`, `gsc_sum_position`, `gsc_clicks`
* **Traffic & Engagement Context:** `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_total_engagement_sec`, `scroll_events`
* **Traffic Sources & AI Signals:** `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`

**4. Excluded Fields (And Why):**
* `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`
* *Reason:* These boolean flags indicate pipeline and integration status rather than search performance. Including them causes noise and data leakage.

In [62]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Features and Schema Setup for Lane 4
feature_names = list(ds.features.keys())

context_keys = ['report_date', 'client_hash_id', 'content_hash_id']
excluded_keys = ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
label_keys = ['ga4_engaged_sessions']

input_features = [f for f in feature_names if f not in context_keys + excluded_keys + label_keys]

print(f"Total Columns: {len(feature_names)}")
print(f"Context Keys ({len(context_keys)}): {context_keys}")
print(f"Selected Target Label ({len(label_keys)}): {label_keys}")
print(f"Excluded Flags ({len(excluded_keys)}): {excluded_keys}")
print(f"Predictive Input Features ({len(input_features)}): {input_features}")

Total Columns: 30
Context Keys (3): ['report_date', 'client_hash_id', 'content_hash_id']
Selected Target Label (1): ['ga4_engaged_sessions']
Excluded Flags (4): ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
Predictive Input Features (22): ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [63]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

# Query to verify zero duplicate primary keys at the grain level
grain_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM df_sample
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
"""

duplicates = duckdb.sql(grain_query).df()
print(f"Duplicate grain records found: {len(duplicates)}")

Duplicate grain records found: 0


In [64]:
import duckdb

# Query missing values for critical columns
missing_query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN report_date IS NULL THEN 1 ELSE 0 END) AS missing_report_date,
    SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_client_id,
    SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_content_id,
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_gsc_clicks,
    SUM(CASE WHEN ga4_engaged_sessions IS NULL THEN 1 ELSE 0 END) AS missing_ga4_engaged_sessions,
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_gsc_impressions
FROM df_sample;
"""

missing_summary = duckdb.sql(missing_query).df()
print(missing_summary.T)

                                    0
total_rows                    50000.0
missing_report_date               0.0
missing_client_id                 0.0
missing_content_id                0.0
missing_gsc_clicks                0.0
missing_ga4_engaged_sessions      0.0
missing_gsc_impressions           0.0


In [65]:
import duckdb

# Query to verify the minimum date, maximum date, and total distinct days
window_query = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS total_days,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_pages
FROM df_sample;
"""

window_summary = duckdb.sql(window_query).df()
print(window_summary)

    min_date   max_date  total_days  total_clients  total_pages
0 2025-01-27 2025-02-27          27              3         5887


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits & Blindspots

**1. Integration & Tracking Gaps:**
* If `client_has_gsc` or `client_has_ga4` is false (or data is unavailable for specific date ranges), organic search performance or on-page behavior cannot be measured, creating missing context or biased scoring.

**2. Off-Page Intent & Uncaptured Demand:**
* The dataset measures active performance on existing pages (`gsc_impressions`, `ga4_pageviews`). It cannot tell us about uncaptured search volume for topics where the client currently has no published content.

**3. Short Time Window (27-Day Horizon):**
* The observed window covers only 27 days (`2025-01-27` to `2025-02-27`). This short duration limits the ability to account for long-term seasonality, day-of-week anomalies, or historical trend baseline shifts.

**4. Aggregated Daily Grain Limitations:**
* Aggregating data at the daily level hides intra-day spikes, specific user session journeys, device breakdowns, and individual search query terms triggering the impressions.

In [66]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Query to check integration availability flags across sample records
limits_query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN client_has_gsc = FALSE THEN 1 ELSE 0 END) AS missing_gsc_integration,
    SUM(CASE WHEN client_has_ga4 = FALSE THEN 1 ELSE 0 END) AS missing_ga4_integration,
    SUM(CASE WHEN gsc_data_available = FALSE THEN 1 ELSE 0 END) AS gsc_unavailable_days,
    SUM(CASE WHEN ga4_data_available = FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_days
FROM df_sample;
"""

limits_summary = duckdb.sql(limits_query).df()
print(limits_summary.T)

                               0
total_rows               50000.0
missing_gsc_integration      0.0
missing_ga4_integration      0.0
gsc_unavailable_days         0.0
ga4_unavailable_days     50000.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.